In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader



transform = transforms.Compose([
    transforms.ToTensor()
])

train_data = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)




class ImprovedCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.3),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)




def train_model(model, use_clip=False):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 10

    loss_history = []

    for epoch in range(epochs):
        model.train()

        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            if use_clip:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

            optimizer.step()

            total_loss += loss.item()

        average_loss = total_loss / len(train_loader)
        loss_history.append(average_loss)

        print(
            f"Епоха {epoch + 1}/{epochs} | "
            f"Loss: {average_loss:.4f}"
        )

    return model, loss_history



def evaluate_model(model):
    model.eval()

    criterion = nn.CrossEntropyLoss()

    correct = 0
    total = 0
    total_loss = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    average_loss = total_loss / len(test_loader)

    return accuracy, average_loss



print("Навчання моделі без нормалізації")
simple_model, simple_losses = train_model(
    SimpleCNN(),
    use_clip=False
)

simple_accuracy, simple_loss = evaluate_model(simple_model)

print("\nНавчання моделі BatchNorm + Dropout")
improved_model, improved_losses = train_model(
    ImprovedCNN(),
    use_clip=True
)

improved_accuracy, improved_loss = evaluate_model(improved_model)



epochs_range = range(1, 11)

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_range,
    simple_losses,
    label="Без нормалізації"
)

plt.plot(
    epochs_range,
    improved_losses,
    label="BatchNorm + Dropout"
)

plt.xlabel("Епоха")
plt.ylabel("Loss")
plt.title("Порівняння втрат за епохами")
plt.legend()
plt.grid(True)

plt.savefig(
    "cnn_batchnorm_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()



results = pd.DataFrame({
    "Конфігурація": [
        "Без нормалізації",
        "BatchNorm + Dropout"
    ],
    "Accuracy": [
        simple_accuracy,
        improved_accuracy
    ],
    "Loss": [
        simple_loss,
        improved_loss
    ],
    "Примітка": [
        "вихідна модель",
        "поліпшена стабільність"
    ]
})

display(results)


print(
    "Висновок:\n"
    "BatchNorm допомагає стабілізувати навчання, "
    "Dropout зменшує ризик перенавчання, "
    "а clip_grad_norm_ обмежує занадто великі градієнти. "
    "Якщо Loss у покращеної моделі зменшується стабільніше, "
    "а Accuracy не падає або зростає, то зміни позитивно вплинули на модель."
)

100.0%
100.0%
100.0%
100.0%


Device: cpu
Навчання моделі без нормалізації
Епоха 1/10 | Loss: 0.2433
Епоха 2/10 | Loss: 0.0609
Епоха 3/10 | Loss: 0.0407
Епоха 4/10 | Loss: 0.0320
Епоха 5/10 | Loss: 0.0242
Епоха 6/10 | Loss: 0.0192
